# 03 — Failure cases

Finds the false positives and false negatives automatically by comparing predictions against the
ground-truth labels on the validation split, and writes annotated images you can paste straight
into `docs/error_analysis.md` and the slide pack.

Colours: **green** = ground truth · **blue** = correct detection · **red** = false positive ·
**orange dashed** = false negative (missed).

Runtime: ~2 min. Requires the dataset (same download cell as `01_Training.ipynb`) and `best.pt`.

## 1. Setup

In [ ]:
!pip install -q ultralytics==8.4.155 roboflow

from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass("Roboflow API key: ")
rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace("caprijopi-hotmail-com").project("construction-safety-gsnvb-oz6um").version(1).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(DATA_YAML)

In [ ]:
# best.pt: either re-use the run from 01_Training, or download the published release asset.
WEIGHTS = "/content/best.pt"      # <- change if your weights are elsewhere
WEIGHTS_URL = ""                  # <- or paste the GitHub Release URL and run the next line

import os
if WEIGHTS_URL and not os.path.exists(WEIGHTS):
    !wget -q -O {WEIGHTS} "{WEIGHTS_URL}"

from ultralytics import YOLO
model = YOLO(WEIGHTS)
NAMES = model.names
print(NAMES)

## 2. Match predictions to ground truth

An IoU of 0.5 is the same threshold the mAP@50 figure uses, so what this cell counts as a miss is
what the metric counts as a miss. Predictions are run at a low confidence so we can also see the
*near* misses — detections the model made but scored below the operating threshold.

In [ ]:
import numpy as np, yaml
from pathlib import Path

CONF_OPERATING = 0.35   # the threshold you actually deploy at
CONF_FLOOR     = 0.10   # look below it to spot near-misses
IOU_MATCH      = 0.50

cfg  = yaml.safe_load(open(DATA_YAML))
root = Path(dataset.location)
val_images = sorted((root / "valid" / "images").glob("*"))
print(len(val_images), "validation images")

def load_gt(img_path, w, h):
    lbl = root / "valid" / "labels" / (img_path.stem + ".txt")
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            if not line.strip():
                continue
            c, cx, cy, bw, bh = line.split()[:5]
            cx, cy, bw, bh = float(cx)*w, float(cy)*h, float(bw)*w, float(bh)*h
            out.append((int(c), cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2))
    return out

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

false_pos, false_neg = [], []

for img_path in val_images:
    r = model.predict(str(img_path), conf=CONF_FLOOR, verbose=False)[0]
    h, w = r.orig_shape
    gts = load_gt(img_path, w, h)

    preds = []
    for box, cls, cf in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        preds.append((int(cls), cf, box))
    preds.sort(key=lambda p: -p[1])

    used = set()
    matched_pred = set()
    for pi, (pc, cf, pbox) in enumerate(preds):
        best_i, best_iou = -1, 0.0
        for gi, (gc, *gbox) in enumerate(gts):
            if gi in used or gc != pc:
                continue
            v = iou(pbox, gbox)
            if v > best_iou:
                best_i, best_iou = gi, v
        if best_iou >= IOU_MATCH:
            used.add(best_i)
            matched_pred.add(pi)

    for pi, (pc, cf, pbox) in enumerate(preds):
        if pi not in matched_pred and cf >= CONF_OPERATING:
            false_pos.append({"img": img_path, "cls": NAMES[pc], "conf": cf, "box": pbox})

    for gi, (gc, *gbox) in enumerate(gts):
        if gi not in used:
            near = max([cf for pc, cf, pb in preds
                        if pc == gc and iou(pb, gbox) >= 0.3] + [0.0])
            false_neg.append({"img": img_path, "cls": NAMES[gc], "box": gbox, "near_conf": near})

print(f"{len(false_pos)} false positives at conf>={CONF_OPERATING}")
print(f"{len(false_neg)} false negatives")

## 3. Which classes are failing

Check this against the per-class recall table in the README — `no-helmet` should dominate the false negatives.

In [ ]:
from collections import Counter

print("FALSE POSITIVES by class")
for k, v in Counter(f["cls"] for f in false_pos).most_common():
    print(f"  {k:12s} {v}")

print("\nFALSE NEGATIVES by class")
for k, v in Counter(f["cls"] for f in false_neg).most_common():
    print(f"  {k:12s} {v}")

print("\nMissed entirely (model scored nothing at all there) vs near-misses (scored below threshold):")
cold = [f for f in false_neg if f["near_conf"] == 0]
warm = [f for f in false_neg if f["near_conf"] > 0]
print(f"  cold misses  {len(cold)}")
print(f"  near misses  {len(warm)}  <- these would be recovered by lowering the threshold")

## 4. Pick the six cases

Highest-confidence false positives are the most embarrassing and the most instructive. For false
negatives we prioritise `no-helmet`, since that is the safety-critical class and the one the README
flags as failing.

In [ ]:
PRIORITY_CLASS = "no-helmet"

fp_pick = sorted(false_pos, key=lambda f: -f["conf"])[:3]

fn_sorted = sorted(false_neg, key=lambda f: (f["cls"] != PRIORITY_CLASS, f["near_conf"]))
fn_pick = fn_sorted[:3]

print("FALSE POSITIVES to screenshot")
for i, f in enumerate(fp_pick, 1):
    print(f"  fp_{i:02d}  {f['cls']:10s} conf {f['conf']:.2f}  {f['img'].name}")

print("\nFALSE NEGATIVES to screenshot")
for i, f in enumerate(fn_pick, 1):
    tag = "cold miss" if f["near_conf"] == 0 else f"near miss (best {f['near_conf']:.2f})"
    print(f"  fn_{i:02d}  {f['cls']:10s} {tag:26s} {f['img'].name}")

## 5. Draw and save

In [ ]:
from PIL import Image, ImageDraw
import shutil

EV = Path("/content/evidence"); shutil.rmtree(EV, ignore_errors=True); EV.mkdir()

def annotate(case, kind, idx):
    im = Image.open(case["img"]).convert("RGB")
    d = ImageDraw.Draw(im)
    r = model.predict(str(case["img"]), conf=CONF_OPERATING, verbose=False)[0]

    # every prediction the model made, in blue
    for box, cls, cf in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        d.rectangle(box, outline=(40, 120, 255), width=2)
        d.text((box[0]+3, box[1]+3), f"{NAMES[int(cls)]} {cf:.2f}", fill=(40, 120, 255))

    # the case itself, highlighted
    if kind == "fp":
        d.rectangle(case["box"], outline=(230, 30, 30), width=5)
        d.text((case["box"][0]+3, case["box"][1]-14),
               f"FALSE POSITIVE: {case['cls']} {case['conf']:.2f}", fill=(230, 30, 30))
    else:
        x1, y1, x2, y2 = case["box"]
        for off in range(0, int(x2-x1), 16):          # dashed top and bottom edge
            d.line([(x1+off, y1), (min(x1+off+8, x2), y1)], fill=(255, 140, 0), width=5)
            d.line([(x1+off, y2), (min(x1+off+8, x2), y2)], fill=(255, 140, 0), width=5)
        d.line([(x1, y1), (x1, y2)], fill=(255, 140, 0), width=5)
        d.line([(x2, y1), (x2, y2)], fill=(255, 140, 0), width=5)
        d.text((x1+3, y1-14), f"MISSED: {case['cls']}", fill=(255, 140, 0))

    out = EV / f"{kind}_{idx:02d}_{case['cls']}.png"
    im.save(out)
    return out

from IPython.display import Image as Show, display

saved = []
for i, f in enumerate(fp_pick, 1):
    saved.append(annotate(f, "fp", i))
for i, f in enumerate(fn_pick, 1):
    saved.append(annotate(f, "fn", i))

for p in saved:
    print(p.name)
    display(Show(str(p), width=640))

## 6. Download

Unzip into `results/evidence/` in the repo, then write the caption for each one in
`docs/error_analysis.md`. The caption is where the marks are — the image only shows *what* failed,
the caption has to say *why*.

In [ ]:
shutil.make_archive("/content/failure_cases", "zip", EV)
from google.colab import files
files.download("/content/failure_cases.zip")